# Validasi Silang (Cross Validation)

Course 2: Supervised Learning


## Dataset

Sama seperti notebook regresi polinomial: `encoded_car_data.csv` (harga mobil dengan fitur yang sudah di-encode), target `price`:

https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/encoded_car_data.csv

## 1. Import dan baca data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.pipeline import Pipeline

In [ ]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/encoded_car_data.csv"
data = pd.read_csv(URL)
print(data.shape)
data.dtypes.value_counts()

Semua kolom sudah numerik, jadi bisa langsung dipakai untuk model.

## 2. Train-test split

In [ ]:
X = data.drop(columns="price")
y = data["price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(X_train.shape, X_test.shape)

Sebanyak 30% data disisihkan untuk test, 70% untuk training. `random_state=42` dipasang supaya hasilnya bisa diulang dengan persis sama.

### Model linear dan evaluasinya

In [ ]:
lr = LinearRegression().fit(X_train, y_train)
pred_test = lr.predict(X_test)

print("R2 train :", round(lr.score(X_train, y_train), 4))
print("R2 test  :", round(lr.score(X_test, y_test), 4))
print("R2 (r2_score) :", round(r2_score(y_test, pred_test), 4))

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, pred_test))
print("RMSE test:", round(rmse, 2))

$R^2$ menunjukkan berapa persen variasi harga yang bisa dijelaskan model, sedangkan RMSE adalah rata-rata besar kesalahan prediksi dalam satuan harga (makin kecil makin baik). $R^2$ di data test biasanya lebih rendah daripada di data train, dan nilai test-lah yang lebih jujur menggambarkan performa model pada data baru.

### Contoh prediksi

In [ ]:
contoh = X.iloc[:3]
pd.DataFrame({"prediksi": lr.predict(contoh).round(0), "sebenarnya": y.iloc[:3].values})

### Dengan pipeline

In [ ]:
pipa = Pipeline([("ss", StandardScaler()), ("lr", LinearRegression())])
pipa.fit(X_train, y_train)

print("R2 train:", round(pipa.score(X_train, y_train), 4))
print("R2 test :", round(pipa.score(X_test, y_test), 4))

`Pipeline` merangkai langkah pemrosesan, di sini standardisasi lalu regresi. Scaler otomatis hanya belajar dari data train.

### Latihan 1: ganti standardisasi dengan normalisasi

Satu catatan: `Normalizer` di scikit-learn menormalkan setiap **baris** (sampel) agar panjang vektornya 1, bukan tiap kolom seperti `MinMaxScaler`. Jadi efeknya berbeda dari scaling fitur biasa.

In [ ]:
pipa_norm = Pipeline([("nn", Normalizer()), ("lr", LinearRegression())])
pipa_norm.fit(X_train, y_train)

rmse_norm = np.sqrt(mean_squared_error(y_test, pipa_norm.predict(X_test)))
print("R2 train:", round(pipa_norm.score(X_train, y_train), 4))
print("R2 test :", round(pipa_norm.score(X_test, y_test), 4))
print("RMSE test:", round(rmse_norm, 2))

## 3. Memilih satu fitur terbaik

Aku latih model dengan satu fitur saja secara bergantian. Fitur terbaik dipilih berdasarkan $R^2$ di **data train**. Kalau dipilih berdasarkan data test, data test ikut dipakai untuk memilih sehingga nilainya tidak lagi murni. Nilai test tetap kuhitung, tapi hanya untuk melihat apakah pilihannya masuk akal.

In [ ]:
r2_fitur = pd.DataFrame(index=X.columns, columns=["train", "test"], dtype=float)
for f in X.columns:
    m = Pipeline([("ss", StandardScaler()), ("lr", LinearRegression())])
    m.fit(X_train[[f]], y_train)
    r2_fitur.loc[f, "train"] = m.score(X_train[[f]], y_train)
    r2_fitur.loc[f, "test"] = m.score(X_test[[f]], y_test)

r2_fitur["train"].plot.bar(figsize=(12, 3.5))
plt.ylabel("R2 (train)")
plt.show()

terbaik = r2_fitur["train"].idxmax()
print("fitur terbaik:", terbaik)
print(r2_fitur.loc[terbaik].round(3))

### Latihan 2: fitur terbaik tanpa standardisasi

In [ ]:
r2_tanpa_ss = pd.DataFrame(index=X.columns, columns=["train", "test"], dtype=float)
for f in X.columns:
    m = LinearRegression().fit(X_train[[f]], y_train)
    r2_tanpa_ss.loc[f, "train"] = m.score(X_train[[f]], y_train)
    r2_tanpa_ss.loc[f, "test"] = m.score(X_test[[f]], y_test)

terbaik2 = r2_tanpa_ss["train"].idxmax()
print("fitur terbaik tanpa standardisasi:", terbaik2)
print(r2_tanpa_ss.loc[terbaik2].round(3))
print("sama dengan yang pakai standardisasi:", terbaik2 == terbaik)

Hasilnya sama karena untuk regresi linear satu fitur, standardisasi hanya mengubah skala sumbu dan tidak mengubah garis terbaiknya.

## 4. K-Fold cross validation

Di K-fold, data dibagi menjadi K bagian (lipatan). Model dilatih K kali; setiap putaran satu lipatan jadi data uji dan sisanya jadi data latih. Hasilnya K skor, dan rata-ratanya jadi perkiraan performa yang lebih stabil dibanding satu kali split.

### `cross_val_score`

In [ ]:
skor = cross_val_score(LinearRegression(), X, y, scoring="r2", cv=3)

def ringkas(skor):
    print("skor per lipatan:", np.round(skor, 4))
    print("rata-rata       :", round(skor.mean(), 4))
    print("standar deviasi :", round(skor.std(), 4))

ringkas(skor)

Aku coba jumlah lipatan yang berbeda untuk melihat pengaruhnya:

In [ ]:
for k in [3, 5, 10]:
    s = cross_val_score(LinearRegression(), X, y, scoring="r2", cv=k)
    print(f"cv={k:<2} rata-rata R2 = {s.mean():.4f} | std = {s.std():.4f}")

Makin banyak lipatan, tiap putaran memakai lebih banyak data untuk training, tapi waktu hitungnya juga bertambah.

### Latihan 3: 5 lipatan dengan `neg_mean_squared_error`

Skor di scikit-learn didefinisikan "makin besar makin baik", jadi untuk error dipakai versi negatifnya. Untuk dapat RMSE, tanda minus dibuang dulu lalu diakar.

In [ ]:
neg_mse = cross_val_score(LinearRegression(), X, y, scoring="neg_mean_squared_error", cv=5)
rmse_cv = np.sqrt(-neg_mse)
ringkas(rmse_cv)

### `KFold` manual

Untuk model yang tidak tersedia di scikit-learn atau data yang besar, kita bisa memakai `KFold` untuk mendapatkan indeks train dan test, lalu menulis loop sendiri.

In [ ]:
def r2_kfold(n_splits, shuffle=False, seed=None):
    kf = KFold(n_splits=n_splits, shuffle=shuffle, random_state=seed if shuffle else None)
    hasil = []
    for idx_tr, idx_te in kf.split(X):
        model = Pipeline([("ss", StandardScaler()), ("lr", LinearRegression())])
        model.fit(X.iloc[idx_tr], y.iloc[idx_tr])
        hasil.append(model.score(X.iloc[idx_te], y.iloc[idx_te]))
    return np.array(hasil)

for n in [2, 3]:
    s = r2_kfold(n)
    print(f"{n} lipatan -> R2 per lipatan {np.round(s, 3)} | rata-rata {s.mean():.4f}")

Dengan 2 lipatan, separuh data dipakai untuk melatih. Dengan 3 lipatan, dua pertiga data dipakai melatih.

### Latihan 4: acak data sebelum dibagi (`shuffle=True`)

Tanpa shuffle, lipatan diambil berurutan sesuai urutan baris. Kalau datanya terurut (misalnya per merek), satu lipatan bisa berisi jenis data yang sangat berbeda dari lipatan lain. Dengan `shuffle=True` barisnya diacak dulu.

In [ ]:
s_acak = r2_kfold(3, shuffle=True, seed=42)
print("tanpa shuffle:", np.round(r2_kfold(3), 3), "| rata-rata", round(r2_kfold(3).mean(), 4))
print("dengan shuffle:", np.round(s_acak, 3), "| rata-rata", round(s_acak.mean(), 4))